# Cross-Modal Diagnostic Observability — Stage T2-J

## Expansion harmonisation, cross-roster deduplication and public-route repair v0.1

This notebook performs the full next step in one run:

- harmonises MILK10K to dermoscopic melanoma-versus-nevus;
- harmonises BrEaST-Lesions-USG to malignant-versus-benign;
- computes exact pixel and perceptual fingerprints;
- deduplicates against the frozen dermoscopy and breast-ultrasound development rosters;
- creates deterministic grouped split manifests;
- excludes BUS-UCLM as an already-used development dataset alias;
- attempts an official HiSBreast public-route repair;
- creates exact manual download folders and instructions when automation is not legally or technically available.

It does **not** calculate AUC, fit source axes, refit RA-CB, inspect locked-blind data, or authorise Stage 12.

Use a clean Colab CPU runtime and select **Runtime → Run all**. The dermoscopy cross-roster fingerprint pass can take several minutes.


In [1]:
# @title T2-J-0. Mount Drive, verify immutable parents and seal the protocol
import base64, hashlib, html, io, json, math, os, re, shutil, tarfile, time, warnings, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps
from scipy.fft import dctn
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability") if IN_COLAB else Path.cwd()
PROJECT_ROOT = Path(os.environ.get("CDO_PROJECT_ROOT", str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
STUDY_ROOT = PROJECT_ROOT / "04_Study_Design"
CM_ROOT = PROJECT_ROOT / "06_Data_Records" / "Cross_Modal"
ACQ_ROOT = PROJECT_ROOT / "00_Data_Acquisition" / "Cross_Modal_Independent_Target_Expansion_v0.1"
RESULT_ROOT = CM_ROOT / "StageT2-J_Expansion_Harmonisation_Dedup_And_Public_Route_Repair_v0.1"
P0, P1, P2, P3, P4, P5 = [RESULT_ROOT / x for x in [
    "00_Protocol", "01_Route_Repair_And_Manual_Queue", "02_Harmonised_Manifests",
    "03_Fingerprints_And_Dedup", "04_Grouped_Splits", "05_Results"
]]
for p in [CODE_ROOT, STUDY_ROOT, ACQ_ROOT, P0, P1, P2, P3, P4, P5]:
    p.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = "CrossModal_StageT2-J_Expansion_Harmonisation_Dedup_And_Public_Route_Repair_v0.1.ipynb"
NOTEBOOK_PATH = CODE_ROOT / NOTEBOOK_NAME
PREREG_PATH = STUDY_ROOT / "StageT2-J_Expansion_Harmonisation_Dedup_And_Public_Route_Repair_Preregistration_v1.0.md"
MANUAL_GUIDE_PATH = STUDY_ROOT / "StageT2-J_Manual_Download_Queue_And_Drop_Locations_v0.1.md"

T2I_FINAL = CM_ROOT / "StageT2-I_Independent_Target_Expansion_Registry_Acquisition_And_Harmonisation_v0.1" / "04_Results" / "StageT2-I_Complete_v0.1.json"
T3PF_FINAL = CM_ROOT / "StageT3-PF_Outcome-Free_Preregistration_And_Asset_Preflight_v1.0" / "04_Results" / "StageT3-PF_Activation_Record_v1.0.json"

STAGE8_MANIFEST_ROOT = CM_ROOT / "Stage8_CrossModality_EdgeLibrary_Expansion_v0.1" / "01_Acquisition_Manifests"
HAM_MANIFEST = STAGE8_MANIFEST_ROOT / "HAM10000_Harmonised_Acquisition_Manifest_v0.1.csv"
UDA_MANIFEST = STAGE8_MANIFEST_ROOT / "ISIC_UDA1_Harmonised_Acquisition_Manifest_v0.1.csv"
MSK_MANIFEST = STAGE8_MANIFEST_ROOT / "ISIC_MSK1_Harmonised_Acquisition_Manifest_v0.1.csv"
US_MANIFEST = CM_ROOT / "Stage11D-R_Cross_Roster_Dedup_Exact_Manifest_And_Grouped_Split_Freeze_v0.1" / "01_Exact_Manifest" / "Stage11D-R_Frozen_Exact_Image_Label_Group_Manifest_v0.1.csv"

EXPECTED = {
    "t2i_file_sha256": "11b92f08ac2e1bfe24e90fda4d46d6cf9bd514f68a9067021dcdadec9f40cb97",
    "t2i_record_sha256": "810f4c8380263ba4cdf0cd63e4f4621362718b72ae7890ed603f1a29332d9c05",
    "t3pf_file_sha256": "10646d771a3cd9e86c8c96eb4a134d4878c7542bb4b0b07ab9e01fa8b0c09c25",
    "t3pf_record_sha256": "4397cee7798f684159ed77aa5e1edd7b7ae0a24378047d6c89b37ef9ef738a52",
    "ham_sha256": "2743942fc27e92dc24770f9f58e8c18ff159901adcbaac98f07c6629b08116db",
    "uda_sha256": "4c0fb4aa6966cc55b0a8d359e47daa8a20e76172ec6f88376c402b16ecbd8980",
    "msk_sha256": "df98282397d005e7caa5444c1d734bea5b85724051f34b49dd3b94c55b53581b",
    "us_sha256": "3e66aad296106d12958a8d84634e5cc780399329ba22b503e49f42019bf3db11",
    "prereg_sha256": "1337a882709cf369bbbfa99786638cc384786aecf203ed9d9315cc2b8b2b6bac",
    "manual_sha256": "e7375eae9e204b536ae606e3b5f4fe2032e027b405c3de578d42975be493c327",
}

LOCKED_BLIND = {"BUSI_CAIRO_2019", "OASBUD_2017", "DERM7PT_2019"}
SEED = 20260722
PHASH_MAX_DISTANCE = 4
NEAR_COPY_MIN_CORRELATION = 0.995
MIN_REFERENCE_COVERAGE = 0.95
HTTP_TIMEOUT = (20, 120)
USER_AGENT = "CMDO-StageT2-J/0.1 governed research acquisition"
MAX_HISBREAST_BYTES = int(1.5 * 1024**3)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha_json(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()
    ).hexdigest()

def canonical_csv(frame):
    return frame.fillna("").to_csv(index=False, lineterminator="\n", float_format="%.12g")

def write_csv(path, frame):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(canonical_csv(frame), encoding="utf-8")

def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

def write_text(path, text):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text, encoding="utf-8")

def verify_self(path, field, expected=None):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    claim = value[field]
    core = dict(value)
    core.pop(field)
    assert sha_json(core) == claim, f"Self-hash mismatch: {path}"
    if expected is not None:
        assert claim == expected, f"Unexpected self-hash: {path}"
    return value

def notebook_source_sha(path):
    value = json.loads(Path(path).read_text(encoding="utf-8"))
    cells = []
    for cell in value.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else str(source)
        cells.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha_json(cells)

def seal_protocol(path, payload):
    path = Path(path)
    value = dict(payload)
    if path.exists():
        old = verify_self(path, "protocol_seal_sha256")
        for key, expected_value in payload.items():
            assert old[key] == expected_value, f"Protocol replay mismatch: {key}"
        return old
    value["sealed_utc"] = now()
    value["protocol_seal_sha256"] = sha_json(value)
    write_json(path, value)
    return value

required = [
    NOTEBOOK_PATH, PREREG_PATH, MANUAL_GUIDE_PATH, T2I_FINAL, T3PF_FINAL,
    HAM_MANIFEST, UDA_MANIFEST, MSK_MANIFEST, US_MANIFEST,
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing required files:\n" + "\n".join(missing)

for role, path in {
    "t2i_file_sha256": T2I_FINAL,
    "t3pf_file_sha256": T3PF_FINAL,
    "ham_sha256": HAM_MANIFEST,
    "uda_sha256": UDA_MANIFEST,
    "msk_sha256": MSK_MANIFEST,
    "us_sha256": US_MANIFEST,
    "prereg_sha256": PREREG_PATH,
    "manual_sha256": MANUAL_GUIDE_PATH,
}.items():
    observed = sha_file(path)
    assert observed == EXPECTED[role], f"Hash mismatch {role}: {observed}"

t2i = verify_self(T2I_FINAL, "final_record_sha256", EXPECTED["t2i_record_sha256"])
t3pf = verify_self(T3PF_FINAL, "activation_record_sha256", EXPECTED["t3pf_record_sha256"])
assert t2i["blind_assets_touched"] is False
assert t2i["blind_outcomes_accessed"] is False
assert t2i["stage12_authorised"] is False
assert t3pf["blind_assets_acquired"] is False
assert t3pf["blind_outcomes_accessed"] is False
assert t3pf["stage12_authorised"] is False

protocol = seal_protocol(P0 / "StageT2-J_Protocol_Seal_v0.1.json", {
    "stage": "StageT2-J",
    "purpose": "expansion_harmonisation_cross_roster_dedup_and_public_route_repair",
    "parent_t2i_record": EXPECTED["t2i_record_sha256"],
    "parent_t3pf_record": EXPECTED["t3pf_record_sha256"],
    "preregistration_sha256": EXPECTED["prereg_sha256"],
    "manual_guide_sha256": EXPECTED["manual_sha256"],
    "ham_manifest_sha256": EXPECTED["ham_sha256"],
    "uda_manifest_sha256": EXPECTED["uda_sha256"],
    "msk_manifest_sha256": EXPECTED["msk_sha256"],
    "ultrasound_manifest_sha256": EXPECTED["us_sha256"],
    "notebook_source_sha256": notebook_source_sha(NOTEBOOK_PATH),
    "bus_uclm_alias_excluded": True,
    "target_outcomes_scored": False,
    "locked_blind_assets_touched": False,
    "locked_blind_outcomes_accessed": False,
    "stage12_authorised": False,
})

print("Stage T2-J protocol:", protocol["protocol_seal_sha256"])
print("Locked blind assets touched:", False)


Mounted at /content/drive
Stage T2-J protocol: 3673a198cbf428702d4ce01b06800de1b21e4ef7d5781e70fbe977c87268ebb3
Locked blind assets touched: False


In [2]:
# @title T2-J-1. Create exact manual drop locations and attempt HiSBreast official-route repair
MANUAL_QUEUE = [
    {
        "priority": 1, "dataset_id": "HISBREAST_V2",
        "official_url": "https://data.mendeley.com/datasets/5c723rpwz2/2",
        "required_action": "Download HiSBreast_Version 2.zip from the official page if automatic discovery fails.",
    },
    {
        "priority": 2, "dataset_id": "PH2",
        "official_url": "https://www.fc.up.pt/addi/ph2%20database.html",
        "required_action": "Complete official quick registration and download the released archive.",
    },
    {
        "priority": 3, "dataset_id": "MESSIDOR_ORIGINAL",
        "official_url": "https://www.adcis.net/en/third-party/messidor/",
        "required_action": "Accept/sign the official ADCIS research agreement and download the original release.",
    },
    {
        "priority": 4, "dataset_id": "MESSIDOR2",
        "official_url": "https://www.adcis.net/en/third-party/messidor2/",
        "required_action": "Download official images; diagnostic-label source remains a separate hold.",
    },
    {
        "priority": 5, "dataset_id": "BRSET_V1_0_1",
        "official_url": "https://physionet.org/content/brazilian-ophthalmological/1.0.1/",
        "required_action": "Complete PhysioNet credentialing, training and DUA.",
    },
    {
        "priority": 6, "dataset_id": "mBRSET_V1_0",
        "official_url": "https://physionet.org/content/mbrset/1.0/",
        "required_action": "Complete PhysioNet credentialing, training and DUA.",
    },
    {
        "priority": 7, "dataset_id": "ODIR5K_DR",
        "official_url": "https://odir2019.grand-challenge.org/dataset/",
        "required_action": "Register on the official Grand Challenge page and accept its terms.",
    },
    {
        "priority": 8, "dataset_id": "FGADR_SEG",
        "official_url": "https://csyizhou.github.io/FGADR/",
        "required_action": "Submit the official non-commercial research agreement.",
    },
    {
        "priority": 9, "dataset_id": "UDIAT_DATASET_B",
        "official_url": "https://helward.mmu.ac.uk/STAFF/M.Yap/dataset.php",
        "required_action": "Request access from the official dataset author.",
    },
]

queue_rows = []
for item in MANUAL_QUEUE:
    raw_inbox = ACQ_ROOT / item["dataset_id"] / "00_Raw_Inbox"
    raw_inbox.mkdir(parents=True, exist_ok=True)
    exact_drive_path = str(raw_inbox)
    instruction = (
        f"Dataset: {item['dataset_id']}\n"
        f"Official route: {item['official_url']}\n"
        f"Required action: {item['required_action']}\n"
        f"Place the original, unmodified archive in this folder:\n{exact_drive_path}\n"
        "Do not extract manually. Rerun Stage T2-J after the file is present.\n"
    )
    write_text(raw_inbox / "MANUAL_DROP_HERE.txt", instruction)
    queue_rows.append({**item, "exact_drive_drop_folder": exact_drive_path})

manual_queue = pd.DataFrame(queue_rows).sort_values("priority")
write_csv(P1 / "StageT2-J_Manual_Download_Queue_v0.1.csv", manual_queue)

# Explicit registry readjudication: BUS-UCLM is already a frozen development source.
readjudication = pd.DataFrame([
    {
        "dataset_id": "BUS_UCLM_V3",
        "readjudicated_status": "EXCLUDE_EXISTING_DEVELOPMENT_DATASET_ALIAS",
        "canonical_existing_dataset": "BUS_UCLM_2025_V3",
        "may_increase_target_level_n": False,
        "reason": "Same official BUS-UCLM release already present in the frozen Stage11D-R source roster.",
    },
    {
        "dataset_id": "BUSC_RODRIGUES_DERIVATIVE",
        "readjudicated_status": "EXCLUDE_DERIVATIVE_DUPLICATE",
        "canonical_existing_dataset": "RODRIGUES_BUI_2017",
        "may_increase_target_level_n": False,
        "reason": "Derivative of the existing Rodrigues release.",
    },
])
write_csv(P1 / "StageT2-J_Registry_Readjudication_v0.1.csv", readjudication)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

def recursively_collect_urls(value, output):
    if isinstance(value, dict):
        for child in value.values():
            recursively_collect_urls(child, output)
    elif isinstance(value, list):
        for child in value:
            recursively_collect_urls(child, output)
    elif isinstance(value, str):
        text = html.unescape(value).replace("\\u002F", "/").replace("\\/", "/")
        if text.startswith("http://") or text.startswith("https://"):
            output.add(text)

def discover_official_asset_urls(page_url):
    urls = set()
    audit = {"page_url": page_url, "status": "", "http_status": None, "error": ""}
    try:
        response = session.get(page_url, timeout=HTTP_TIMEOUT)
        audit["http_status"] = int(response.status_code)
        response.raise_for_status()
        text = html.unescape(response.text).replace("\\u002F", "/").replace("\\/", "/")
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup.find_all("a", href=True):
            urls.add(urljoin(response.url, tag["href"]))
        for script in soup.find_all("script"):
            content = script.string or script.get_text() or ""
            if not content.strip():
                continue
            try:
                recursively_collect_urls(json.loads(content), urls)
            except Exception:
                pass
        for match in re.findall(r'https?://[^"\'<>\s]+', text):
            urls.add(match.rstrip("\\\\,;"))
        audit["status"] = "PAGE_PARSED"
    except Exception as exc:
        audit["status"] = "PAGE_FETCH_FAILED"
        audit["error"] = f"{type(exc).__name__}: {exc}"
    filtered = []
    for url in urls:
        clean = unquote(url)
        lower = clean.lower()
        if any(token in lower for token in [
            ".zip", ".rar", ".7z", "file_download", "downloads.mendeley",
            "datasets-cache", "public-files/datasets",
        ]):
            filtered.append(clean)
    return list(dict.fromkeys(sorted(filtered))), audit

def stream_download(url, destination, max_bytes):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 1024:
        return {
            "status": "ALREADY_PRESENT", "url": url,
            "final_url": url, "bytes": destination.stat().st_size,
            "sha256": sha_file(destination), "error": "",
        }
    temp = destination.with_suffix(destination.suffix + ".part")
    try:
        with session.get(url, timeout=HTTP_TIMEOUT, allow_redirects=True, stream=True) as response:
            response.raise_for_status()
            content_type = response.headers.get("content-type", "").lower()
            if "text/html" in content_type:
                raise RuntimeError("Official candidate returned HTML instead of an archive")
            total = 0
            digest = hashlib.sha256()
            with temp.open("wb") as f:
                for chunk in response.iter_content(1024 * 1024):
                    if not chunk:
                        continue
                    total += len(chunk)
                    if total > max_bytes:
                        raise RuntimeError("Fixed download size cap exceeded")
                    f.write(chunk)
                    digest.update(chunk)
            if total < 1024:
                raise RuntimeError("Downloaded object is unexpectedly small")
            temp.replace(destination)
            return {
                "status": "DOWNLOADED", "url": url,
                "final_url": str(response.url), "bytes": total,
                "sha256": digest.hexdigest(), "error": "",
            }
    except Exception as exc:
        if temp.exists():
            temp.unlink()
        return {
            "status": "DOWNLOAD_FAILED", "url": url,
            "final_url": url, "bytes": 0, "sha256": "",
            "error": f"{type(exc).__name__}: {exc}",
        }

his_inbox = ACQ_ROOT / "HISBREAST_V2" / "00_Raw_Inbox"
existing_his_archives = [
    p for p in his_inbox.iterdir()
    if p.is_file() and p.suffix.lower() in {".zip", ".rar", ".7z", ".tar", ".gz"}
]
his_route_rows = []

if existing_his_archives:
    for path in existing_his_archives:
        his_route_rows.append({
            "dataset_id": "HISBREAST_V2", "candidate_url": "MANUAL_OFFICIAL_INBOX",
            "status": "MANUAL_FILE_PRESENT", "filename": path.name,
            "bytes": path.stat().st_size, "sha256": sha_file(path), "error": "",
        })
else:
    candidates, page_audit = discover_official_asset_urls(
        "https://data.mendeley.com/datasets/5c723rpwz2/2"
    )
    write_json(P1 / "StageT2-J_HiSBreast_Page_Discovery_Audit_v0.1.json", {
        **page_audit, "candidate_urls": candidates,
    })
    # Prefer one archive labelled as Version 2; otherwise try plausible ZIP candidates in order.
    candidates = sorted(
        candidates,
        key=lambda u: (
            0 if ("version%202" in u.lower() or "version_2" in u.lower() or "version-2" in u.lower()) else 1,
            0 if ".zip" in u.lower() else 1,
            len(u),
        ),
    )
    selected_success = False
    for index, url in enumerate(candidates[:12]):
        parsed_name = Path(urlparse(url).path).name
        filename = parsed_name if parsed_name.lower().endswith(".zip") else f"HiSBreast_official_candidate_{index+1}.zip"
        result = stream_download(url, his_inbox / filename, MAX_HISBREAST_BYTES)
        his_route_rows.append({
            "dataset_id": "HISBREAST_V2", "candidate_url": url,
            "filename": filename, **result,
        })
        if result["status"] in {"DOWNLOADED", "ALREADY_PRESENT"}:
            selected_success = True
            break
    if not selected_success:
        his_route_rows.append({
            "dataset_id": "HISBREAST_V2",
            "candidate_url": "https://data.mendeley.com/datasets/5c723rpwz2/2",
            "filename": "",
            "status": "HOLD_MANUAL_OFFICIAL_DOWNLOAD_REQUIRED",
            "bytes": 0, "sha256": "",
            "error": "Use the official page and place HiSBreast_Version 2.zip in the exact raw inbox.",
        })

his_route_receipts = pd.DataFrame(his_route_rows)
write_csv(P1 / "StageT2-J_HiSBreast_Route_Repair_Receipts_v0.1.csv", his_route_receipts)

display(readjudication)
display(his_route_receipts)
display(manual_queue[["priority", "dataset_id", "exact_drive_drop_folder"]])


,dataset_id,readjudicated_status,canonical_existing_dataset,may_increase_target_level_n,reason
0,BUS_UCLM_V3,EXCLUDE_EXISTING_DEVELOPMENT_DATASET_ALIAS,BUS_UCLM_2025_V3,False,Same official BUS-UCLM release already present...
1,BUSC_RODRIGUES_DERIVATIVE,EXCLUDE_DERIVATIVE_DUPLICATE,RODRIGUES_BUI_2017,False,Derivative of the existing Rodrigues release.


,dataset_id,candidate_url,filename,status,bytes,sha256,error
0,HISBREAST_V2,https://data.mendeley.com/datasets/5c723rpwz2/2,,HOLD_MANUAL_OFFICIAL_DOWNLOAD_REQUIRED,0,,Use the official page and place HiSBreast_Vers...


,priority,dataset_id,exact_drive_drop_folder
0,1,HISBREAST_V2,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,2,PH2,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,3,MESSIDOR_ORIGINAL,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,4,MESSIDOR2,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,5,BRSET_V1_0_1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,6,mBRSET_V1_0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,7,ODIR5K_DR,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
7,8,FGADR_SEG,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
8,9,UDIAT_DATASET_B,/content/drive/MyDrive/Cross-Modal_Diagnostic_...


In [3]:
# @title T2-J-2. Safe extraction and reproducible image fingerprint utilities
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def safe_extract_archive(source, destination):
    source = Path(source)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    suffixes = "".join(source.suffixes).lower()
    if suffixes.endswith(".zip"):
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not (target == base or str(target).startswith(str(base) + os.sep)):
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        return "EXTRACTED_ZIP"
    if suffixes.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(source) as archive:
            for member in archive.getmembers():
                target = (destination / member.name).resolve()
                if not (target == base or str(target).startswith(str(base) + os.sep)):
                    raise RuntimeError(f"Unsafe TAR member: {member.name}")
            archive.extractall(destination)
        return "EXTRACTED_TAR"
    return "NOT_SUPPORTED_ARCHIVE"

def ensure_extracted(dataset_id):
    dataset_root = ACQ_ROOT / dataset_id
    inbox = dataset_root / "00_Raw_Inbox"
    extracted = dataset_root / "01_Extracted"
    extracted.mkdir(parents=True, exist_ok=True)
    image_count = sum(1 for p in extracted.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    audit = []
    if image_count > 0:
        return extracted, [{"dataset_id": dataset_id, "archive": "", "status": "ALREADY_EXTRACTED", "images_found": image_count, "error": ""}]
    for archive in sorted(inbox.iterdir()):
        if not archive.is_file() or archive.name == "MANUAL_DROP_HERE.txt":
            continue
        try:
            status = safe_extract_archive(archive, extracted)
            images_found = sum(1 for p in extracted.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
            audit.append({
                "dataset_id": dataset_id, "archive": archive.name,
                "status": status, "images_found": images_found, "error": "",
            })
        except Exception as exc:
            audit.append({
                "dataset_id": dataset_id, "archive": archive.name,
                "status": "EXTRACTION_FAILED", "images_found": 0,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return extracted, audit

def image_fingerprint_from_bytes(data, include_thumb=True):
    with Image.open(io.BytesIO(data)) as image0:
        image = ImageOps.exif_transpose(image0).convert("RGB")
        width, height = image.size
        rgb = np.asarray(image, dtype=np.uint8)
        header = (
            json.dumps(
                {"width": width, "height": height, "mode": "RGB"},
                sort_keys=True, separators=(",", ":"),
            ).encode() + b"\\0"
        )
        pixel_sha = sha_bytes(header + rgb.tobytes(order="C"))
        gray = np.asarray(
            image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),
            dtype=np.float32,
        )
        low = dctn(gray, type=2, norm="ortho")[:8, :8]
        median = float(np.median(low.flatten()[1:]))
        bits = (low.flatten() > median).astype(np.uint8)
        phash = 0
        for bit in bits:
            phash = (phash << 1) | int(bit)
        thumb_b64 = ""
        if include_thumb:
            thumb = np.asarray(
                image.convert("L").resize((32, 32), Image.Resampling.BILINEAR),
                dtype=np.uint8,
            )
            thumb_b64 = base64.b64encode(thumb.tobytes(order="C")).decode("ascii")
        return {
            "pixel_sha256": pixel_sha,
            "pixel_width": int(width),
            "pixel_height": int(height),
            "pixel_mode": "RGB",
            "phash64": f"{phash:016x}",
            "thumb32_b64": thumb_b64,
            "decode_ok": True,
        }

def image_fingerprint_from_path(path):
    path = Path(path)
    data = path.read_bytes()
    result = image_fingerprint_from_bytes(data, include_thumb=True)
    result["raw_image_sha256"] = sha_bytes(data)
    result["file_size_bytes"] = len(data)
    return result

def decode_thumb(value):
    array = np.frombuffer(base64.b64decode(value), dtype=np.uint8).astype(np.float32)
    array = array.reshape(32, 32).ravel()
    return (array - array.mean()) / (array.std() + 1e-8)

def phash_distance(a, b):
    return (int(str(a), 16) ^ int(str(b), 16)).bit_count()

def thumbnail_correlation(a_b64, b_b64):
    a = decode_thumb(a_b64)
    b = decode_thumb(b_b64)
    return float(np.dot(a, b) / len(a))

def list_image_paths(root):
    return sorted(
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

extraction_audit = []
milk_extracted, audit = ensure_extracted("ISIC_MILK10K")
extraction_audit.extend(audit)
breast_extracted, audit = ensure_extracted("BREAST_LESIONS_USG")
extraction_audit.extend(audit)
his_extracted, audit = ensure_extracted("HISBREAST_V2")
extraction_audit.extend(audit)

extraction_audit = pd.DataFrame(extraction_audit)
write_csv(P2 / "StageT2-J_Extraction_Audit_v0.1.csv", extraction_audit)
display(extraction_audit)


,dataset_id,archive,status,images_found,error
0,ISIC_MILK10K,,ALREADY_EXTRACTED,10480,
1,BREAST_LESIONS_USG,,ALREADY_EXTRACTED,522,


In [4]:
# @title T2-J-3. Harmonise and cross-roster deduplicate MILK10K
milk_root = ACQ_ROOT / "ISIC_MILK10K"
milk_inbox = milk_root / "00_Raw_Inbox"

metadata_path = milk_inbox / "MILK10k_Training_Metadata.csv"
groundtruth_path = milk_inbox / "MILK10k_Training_GroundTruth.csv"
supplement_path = milk_inbox / "MILK10k_Training_Supplement.csv"
assert metadata_path.is_file() and groundtruth_path.is_file(), "MILK10K metadata or ground truth missing"

milk_metadata = pd.read_csv(metadata_path)
milk_groundtruth = pd.read_csv(groundtruth_path)
milk_supplement = pd.read_csv(supplement_path) if supplement_path.is_file() else pd.DataFrame()

assert {"lesion_id", "image_type", "isic_id"}.issubset(milk_metadata.columns)
assert {"lesion_id", "MEL", "NV"}.issubset(milk_groundtruth.columns)
assert milk_groundtruth["lesion_id"].is_unique

milk_groundtruth["binary_label"] = np.where(
    milk_groundtruth["MEL"].eq(1.0), 1,
    np.where(milk_groundtruth["NV"].eq(1.0), 0, np.nan),
)
milk_derm = milk_metadata[
    milk_metadata["image_type"].astype(str).str.lower().eq("dermoscopic")
].copy()
milk = milk_derm.merge(
    milk_groundtruth[["lesion_id", "binary_label"]],
    on="lesion_id", how="left", validate="one_to_one",
)
milk["endpoint_status"] = np.where(
    milk["binary_label"].isna(), "EXCLUDE_NON_MELANOMA_NON_NEVUS", "INCLUDE_BINARY_ENDPOINT"
)
milk["released_label"] = np.where(
    milk["binary_label"].eq(1), "melanoma",
    np.where(milk["binary_label"].eq(0), "melanocytic_nevus", "other"),
)

image_map = {}
for path in list_image_paths(milk_extracted):
    image_map.setdefault(path.stem, path)
milk["image_path"] = milk["isic_id"].map(image_map)
milk["image_present"] = milk["image_path"].notna()

existing_derm = pd.concat(
    [pd.read_csv(HAM_MANIFEST), pd.read_csv(UDA_MANIFEST), pd.read_csv(MSK_MANIFEST)],
    ignore_index=True,
)
existing_derm["existing_lesion_id"] = existing_derm["unit_id"].astype(str).str.extract(r"::LESION::(.+)$")[0]
existing_image_ids = set(existing_derm["image_id"].astype(str))
existing_lesion_labels = (
    existing_derm.dropna(subset=["existing_lesion_id"])
    .groupby("existing_lesion_id")["label"]
    .agg(lambda x: sorted(set(int(v) for v in x)))
    .to_dict()
)

def identifier_status(row):
    if row["endpoint_status"] != "INCLUDE_BINARY_ENDPOINT":
        return row["endpoint_status"]
    image_overlap = str(row["isic_id"]) in existing_image_ids
    lesion_overlap = str(row["lesion_id"]) in existing_lesion_labels
    if not image_overlap and not lesion_overlap:
        return "KEEP_PENDING_FINGERPRINT"
    labels = existing_lesion_labels.get(str(row["lesion_id"]), [])
    if labels and int(row["binary_label"]) not in labels:
        return "QUARANTINE_IDENTIFIER_LABEL_CONFLICT"
    return "EXCLUDE_EXISTING_IDENTIFIER_OVERLAP"

milk["dedup_status"] = milk.apply(identifier_status, axis=1)

fingerprint_rows = []
to_fingerprint = milk[
    milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & milk["image_present"]
].copy()
for index, row in to_fingerprint.iterrows():
    try:
        fp = image_fingerprint_from_path(row["image_path"])
        fingerprint_rows.append({"row_index": int(index), **fp, "fingerprint_error": ""})
    except Exception as exc:
        fingerprint_rows.append({
            "row_index": int(index), "decode_ok": False,
            "fingerprint_error": f"{type(exc).__name__}: {exc}",
        })
    if len(fingerprint_rows) % 250 == 0:
        print("MILK fingerprints:", len(fingerprint_rows), "/", len(to_fingerprint))

milk_fp = pd.DataFrame(fingerprint_rows).set_index("row_index") if fingerprint_rows else pd.DataFrame()
for column in [
    "raw_image_sha256", "pixel_sha256", "pixel_width", "pixel_height", "pixel_mode",
    "phash64", "thumb32_b64", "decode_ok", "file_size_bytes", "fingerprint_error",
]:
    milk[column] = milk_fp[column] if column in milk_fp.columns else np.nan

milk.loc[
    milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & ~milk["image_present"],
    "dedup_status"
] = "HOLD_IMAGE_MISSING"
milk.loc[
    milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & ~milk["decode_ok"].fillna(False),
    "dedup_status"
] = "HOLD_IMAGE_DECODE_FAILURE"

# Internal exact-pixel duplicates.
pending = milk[milk["dedup_status"].eq("KEEP_PENDING_FINGERPRINT")].copy()
for pixel_sha, group in pending.groupby("pixel_sha256", dropna=True):
    if len(group) < 2:
        continue
    ordered = group.sort_values(["lesion_id", "isic_id"])
    labels = sorted(ordered["binary_label"].astype(int).unique())
    if len(labels) > 1:
        milk.loc[ordered.index, "dedup_status"] = "QUARANTINE_INTERNAL_EXACT_LABEL_CONFLICT"
    else:
        milk.loc[ordered.index[1:], "dedup_status"] = "EXCLUDE_INTERNAL_EXACT_PIXEL_DUPLICATE"

# Build or extend the existing dermoscopy fingerprint cache.
reference_cache_path = P3 / "StageT2-J_Existing_Dermoscopy_Reference_Fingerprints_v0.1.csv"
if reference_cache_path.is_file():
    reference_cache = pd.read_csv(reference_cache_path)
else:
    reference_cache = pd.DataFrame()

cached_ids = set(reference_cache["image_id"].astype(str)) if len(reference_cache) else set()
reference_records = existing_derm[
    ~existing_derm["image_id"].astype(str).isin(cached_ids)
][["dataset", "image_id", "unit_id", "group_id", "label", "source_locator"]].to_dict("records")

def fingerprint_reference_url(record):
    output = {
        "dataset": record["dataset"], "image_id": record["image_id"],
        "unit_id": record["unit_id"], "group_id": record["group_id"],
        "label": int(record["label"]), "source_locator": record["source_locator"],
        "status": "", "error": "",
    }
    try:
        response = requests.get(
            record["source_locator"], headers={"User-Agent": USER_AGENT},
            timeout=HTTP_TIMEOUT,
        )
        response.raise_for_status()
        output.update(image_fingerprint_from_bytes(response.content, include_thumb=True))
        output["status"] = "FINGERPRINTED"
    except Exception as exc:
        output["status"] = "DOWNLOAD_OR_DECODE_FAILED"
        output["error"] = f"{type(exc).__name__}: {exc}"
    return output

new_reference_rows = []
if reference_records:
    print("Fingerprinting missing existing dermoscopy references:", len(reference_records))
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(fingerprint_reference_url, record) for record in reference_records]
        for count, future in enumerate(as_completed(futures), 1):
            new_reference_rows.append(future.result())
            if count % 250 == 0:
                print("Reference fingerprints:", count, "/", len(reference_records))

if new_reference_rows:
    reference_cache = pd.concat(
        [reference_cache, pd.DataFrame(new_reference_rows)], ignore_index=True
    )
    reference_cache = reference_cache.drop_duplicates("image_id", keep="last")
write_csv(reference_cache_path, reference_cache)

reference_ok = reference_cache[
    reference_cache["status"].eq("FINGERPRINTED")
    & reference_cache["decode_ok"].fillna(False)
].copy()
reference_coverage = len(reference_ok) / len(existing_derm)

# Cross-roster exact and high-confidence near-copy adjudication.
reference_pixel = {}
for row in reference_ok.itertuples():
    reference_pixel.setdefault(row.pixel_sha256, []).append(row)

candidate_indices = milk.index[milk["dedup_status"].eq("KEEP_PENDING_FINGERPRINT")].tolist()
near_rows = []
for count, index in enumerate(candidate_indices, 1):
    row = milk.loc[index]
    exact = reference_pixel.get(row["pixel_sha256"], [])
    if exact:
        labels = sorted(set(int(item.label) for item in exact))
        milk.loc[index, "dedup_status"] = (
            "QUARANTINE_CROSS_ROSTER_EXACT_LABEL_CONFLICT"
            if int(row["binary_label"]) not in labels
            else "EXCLUDE_CROSS_ROSTER_EXACT_PIXEL_DUPLICATE"
        )
        near_rows.append({
            "candidate_isic_id": row["isic_id"],
            "candidate_lesion_id": row["lesion_id"],
            "candidate_label": int(row["binary_label"]),
            "reference_image_id": exact[0].image_id,
            "reference_dataset": exact[0].dataset,
            "reference_label": int(exact[0].label),
            "phash_distance": 0,
            "thumbnail_correlation": 1.0,
            "action": milk.loc[index, "dedup_status"],
        })
        continue

    best = None
    candidate_hash = int(str(row["phash64"]), 16)
    for ref in reference_ok.itertuples():
        distance = (candidate_hash ^ int(str(ref.phash64), 16)).bit_count()
        if distance > PHASH_MAX_DISTANCE:
            continue
        correlation = thumbnail_correlation(row["thumb32_b64"], ref.thumb32_b64)
        if correlation >= NEAR_COPY_MIN_CORRELATION:
            score = (distance, -correlation, str(ref.dataset), str(ref.image_id))
            if best is None or score < best[0]:
                best = (score, ref, correlation)
    if best is not None:
        ref = best[1]
        correlation = best[2]
        action = (
            "QUARANTINE_CROSS_ROSTER_NEAR_COPY_LABEL_CONFLICT"
            if int(row["binary_label"]) != int(ref.label)
            else "EXCLUDE_CROSS_ROSTER_HIGH_CONFIDENCE_NEAR_COPY"
        )
        milk.loc[index, "dedup_status"] = action
        near_rows.append({
            "candidate_isic_id": row["isic_id"],
            "candidate_lesion_id": row["lesion_id"],
            "candidate_label": int(row["binary_label"]),
            "reference_image_id": ref.image_id,
            "reference_dataset": ref.dataset,
            "reference_label": int(ref.label),
            "phash_distance": best[0][0],
            "thumbnail_correlation": correlation,
            "action": action,
        })
    if count % 250 == 0:
        print("MILK cross-roster comparisons:", count, "/", len(candidate_indices))

if reference_coverage < MIN_REFERENCE_COVERAGE:
    milk.loc[
        milk["dedup_status"].eq("KEEP_PENDING_FINGERPRINT"),
        "dedup_status"
    ] = "HOLD_INCOMPLETE_REFERENCE_FINGERPRINT_COVERAGE"
else:
    milk.loc[
        milk["dedup_status"].eq("KEEP_PENDING_FINGERPRINT"),
        "dedup_status"
    ] = "KEEP_UNIQUE"

milk["dataset"] = "ISIC_MILK10K"
milk["modality"] = "dermoscopy"
milk["task"] = "melanoma_vs_melanocytic_nevus"
milk["image_id"] = milk["isic_id"]
milk["unit_id"] = "ISIC_MILK10K::LESION::" + milk["lesion_id"].astype(str)
milk["group_id"] = milk["unit_id"]
milk["source_locator"] = milk["image_path"].astype(str)
milk["source_kind"] = "local_official_archive"
milk["label"] = milk["binary_label"]

milk_output_columns = [
    "dataset", "modality", "task", "image_id", "unit_id", "group_id", "label",
    "released_label", "endpoint_status", "dedup_status", "source_locator",
    "raw_image_sha256", "pixel_sha256", "pixel_width", "pixel_height",
    "pixel_mode", "phash64", "decode_ok",
]
milk_manifest = milk[milk_output_columns].copy()
write_csv(P2 / "StageT2-J_MILK10K_Harmonised_Exact_Manifest_v0.1.csv", milk_manifest)
write_csv(P3 / "StageT2-J_MILK10K_Cross_Roster_Dedup_Candidates_v0.1.csv", pd.DataFrame(near_rows))

milk_summary = pd.DataFrame([{
    "dataset": "ISIC_MILK10K",
    "released_dermoscopic_images": int(len(milk)),
    "endpoint_eligible": int(milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT").sum()),
    "eligible_negative": int(milk["binary_label"].eq(0).sum()),
    "eligible_positive": int(milk["binary_label"].eq(1).sum()),
    "identifier_overlap_retired": int(milk["dedup_status"].str.contains("IDENTIFIER").sum()),
    "exact_or_near_copy_retired_or_quarantined": int(
        milk["dedup_status"].str.contains("EXACT|NEAR_COPY", regex=True).sum()
    ),
    "retained_unique": int(milk["dedup_status"].eq("KEEP_UNIQUE").sum()),
    "retained_groups": int(milk.loc[milk["dedup_status"].eq("KEEP_UNIQUE"), "group_id"].nunique()),
    "reference_fingerprint_coverage": reference_coverage,
}])
write_csv(P3 / "StageT2-J_MILK10K_Dedup_Summary_v0.1.csv", milk_summary)

display(milk_summary)
print("MILK status counts:")
display(milk["dedup_status"].value_counts().rename_axis("status").reset_index(name="rows"))


MILK fingerprints: 250 / 1196
MILK fingerprints: 500 / 1196
MILK fingerprints: 750 / 1196
MILK fingerprints: 1000 / 1196
Fingerprinting missing existing dermoscopy references: 3747
Reference fingerprints: 250 / 3747
Reference fingerprints: 500 / 3747
Reference fingerprints: 750 / 3747
Reference fingerprints: 1000 / 3747
Reference fingerprints: 1250 / 3747
Reference fingerprints: 1500 / 3747
Reference fingerprints: 1750 / 3747
Reference fingerprints: 2000 / 3747
Reference fingerprints: 2250 / 3747
Reference fingerprints: 2500 / 3747
Reference fingerprints: 2750 / 3747
Reference fingerprints: 3000 / 3747
Reference fingerprints: 3250 / 3747
Reference fingerprints: 3500 / 3747
MILK cross-roster comparisons: 250 / 907
MILK cross-roster comparisons: 500 / 907
MILK cross-roster comparisons: 750 / 907


,dataset,released_dermoscopic_images,endpoint_eligible,eligible_negative,eligible_positive,identifier_overlap_retired,exact_or_near_copy_retired_or_quarantined,retained_unique,retained_groups,reference_fingerprint_coverage
0,ISIC_MILK10K,5240,1196,746,450,289,0,907,907,1.0


MILK status counts:


,status,rows
0,EXCLUDE_NON_MELANOMA_NON_NEVUS,4044
1,KEEP_UNIQUE,907
2,EXCLUDE_EXISTING_IDENTIFIER_OVERLAP,289


In [5]:
# @title T2-J-4. Harmonise and deduplicate BrEaST-Lesions-USG
breast_root = ACQ_ROOT / "BREAST_LESIONS_USG"
breast_inbox = breast_root / "00_Raw_Inbox"
xlsx_candidates = sorted(breast_inbox.glob("*.xlsx"))
assert xlsx_candidates, "BrEaST clinical XLSX missing"
breast_clinical = pd.read_excel(xlsx_candidates[0])

required_columns = {"CaseID", "Image_filename", "Classification"}
assert required_columns.issubset(breast_clinical.columns)

breast = breast_clinical.copy()
classification = breast["Classification"].astype(str).str.strip().str.lower()
breast["binary_label"] = np.where(
    classification.eq("malignant"), 1,
    np.where(classification.eq("benign"), 0, np.nan),
)
breast["released_label"] = classification
breast["endpoint_status"] = np.where(
    breast["binary_label"].isna(), "EXCLUDE_NORMAL_OR_UNKNOWN", "INCLUDE_BINARY_ENDPOINT"
)

breast_image_map = {}
for path in list_image_paths(breast_extracted):
    breast_image_map.setdefault(path.name, path)
breast["image_path"] = breast["Image_filename"].map(breast_image_map)
breast["image_present"] = breast["image_path"].notna()
breast["dedup_status"] = np.where(
    breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT"),
    "KEEP_PENDING_FINGERPRINT",
    breast["endpoint_status"],
)

fingerprint_rows = []
to_fingerprint = breast[
    breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & breast["image_present"]
].copy()
for index, row in to_fingerprint.iterrows():
    try:
        fp = image_fingerprint_from_path(row["image_path"])
        fingerprint_rows.append({"row_index": int(index), **fp, "fingerprint_error": ""})
    except Exception as exc:
        fingerprint_rows.append({
            "row_index": int(index), "decode_ok": False,
            "fingerprint_error": f"{type(exc).__name__}: {exc}",
        })

breast_fp = pd.DataFrame(fingerprint_rows).set_index("row_index") if fingerprint_rows else pd.DataFrame()
for column in [
    "raw_image_sha256", "pixel_sha256", "pixel_width", "pixel_height", "pixel_mode",
    "phash64", "thumb32_b64", "decode_ok", "file_size_bytes", "fingerprint_error",
]:
    breast[column] = breast_fp[column] if column in breast_fp.columns else np.nan

breast.loc[
    breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & ~breast["image_present"],
    "dedup_status"
] = "HOLD_IMAGE_MISSING"
breast.loc[
    breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT") & ~breast["decode_ok"].fillna(False),
    "dedup_status"
] = "HOLD_IMAGE_DECODE_FAILURE"

# Internal exact duplicates.
pending = breast[breast["dedup_status"].eq("KEEP_PENDING_FINGERPRINT")].copy()
for pixel_sha, group in pending.groupby("pixel_sha256", dropna=True):
    if len(group) < 2:
        continue
    ordered = group.sort_values(["CaseID", "Image_filename"])
    labels = sorted(ordered["binary_label"].astype(int).unique())
    if len(labels) > 1:
        breast.loc[ordered.index, "dedup_status"] = "QUARANTINE_INTERNAL_EXACT_LABEL_CONFLICT"
    else:
        breast.loc[ordered.index[1:], "dedup_status"] = "EXCLUDE_INTERNAL_EXACT_PIXEL_DUPLICATE"

# Internal high-confidence near copies.
internal_near_rows = []
pending = breast[breast["dedup_status"].eq("KEEP_PENDING_FINGERPRINT")].copy()
pending_records = list(pending.itertuples())
for i, left in enumerate(pending_records):
    if breast.loc[left.Index, "dedup_status"] != "KEEP_PENDING_FINGERPRINT":
        continue
    for right in pending_records[i + 1:]:
        if breast.loc[right.Index, "dedup_status"] != "KEEP_PENDING_FINGERPRINT":
            continue
        distance = phash_distance(left.phash64, right.phash64)
        if distance > PHASH_MAX_DISTANCE:
            continue
        correlation = thumbnail_correlation(left.thumb32_b64, right.thumb32_b64)
        if correlation < NEAR_COPY_MIN_CORRELATION:
            continue
        labels = {int(left.binary_label), int(right.binary_label)}
        if len(labels) > 1:
            action = "QUARANTINE_INTERNAL_NEAR_COPY_LABEL_CONFLICT"
            breast.loc[[left.Index, right.Index], "dedup_status"] = action
        else:
            action = "EXCLUDE_INTERNAL_HIGH_CONFIDENCE_NEAR_COPY"
            loser = max(left.Index, right.Index)
            breast.loc[loser, "dedup_status"] = action
        internal_near_rows.append({
            "case_id_a": left.CaseID, "case_id_b": right.CaseID,
            "label_a": int(left.binary_label), "label_b": int(right.binary_label),
            "phash_distance": distance, "thumbnail_correlation": correlation,
            "action": action,
        })

existing_us = pd.read_csv(US_MANIFEST)
existing_pixel_labels = (
    existing_us.groupby("pixel_sha256")["binary_label"]
    .agg(lambda x: sorted(set(int(v) for v in x)))
    .to_dict()
)
existing_phash = existing_us[
    existing_us["phash64"].notna() & existing_us["binary_label"].notna()
].copy()

cross_rows = []
for index in breast.index[breast["dedup_status"].eq("KEEP_PENDING_FINGERPRINT")]:
    row = breast.loc[index]
    exact_labels = existing_pixel_labels.get(row["pixel_sha256"], [])
    if exact_labels:
        action = (
            "QUARANTINE_CROSS_ROSTER_EXACT_LABEL_CONFLICT"
            if int(row["binary_label"]) not in exact_labels
            else "EXCLUDE_CROSS_ROSTER_EXACT_PIXEL_DUPLICATE"
        )
        breast.loc[index, "dedup_status"] = action
        cross_rows.append({
            "case_id": row["CaseID"], "reference_sample_id": "",
            "reference_dataset": "", "phash_distance": 0,
            "action": action,
        })
        continue
    candidate_hash = int(str(row["phash64"]), 16)
    best = None
    for ref in existing_phash.itertuples():
        distance = (candidate_hash ^ int(str(ref.phash64), 16)).bit_count()
        if distance <= PHASH_MAX_DISTANCE:
            score = (distance, str(ref.dataset_id), str(ref.sample_id))
            if best is None or score < best[0]:
                best = (score, ref)
    if best is not None:
        ref = best[1]
        action = "HOLD_CROSS_ROSTER_PHASH_NEAR_COPY_REVIEW"
        breast.loc[index, "dedup_status"] = action
        cross_rows.append({
            "case_id": row["CaseID"], "reference_sample_id": ref.sample_id,
            "reference_dataset": ref.dataset_id,
            "phash_distance": best[0][0], "action": action,
        })

breast.loc[
    breast["dedup_status"].eq("KEEP_PENDING_FINGERPRINT"),
    "dedup_status"
] = "KEEP_UNIQUE"

breast["dataset"] = "BREAST_LESIONS_USG"
breast["modality"] = "breast_ultrasound"
breast["task"] = "breast_lesion_malignant_vs_benign"
breast["image_id"] = breast["Image_filename"].astype(str).str.replace(r"\.[^.]+$", "", regex=True)
breast["unit_id"] = "BREAST_LESIONS_USG::CASE::" + breast["CaseID"].astype(str)
breast["group_id"] = breast["unit_id"]
breast["source_locator"] = breast["image_path"].astype(str)
breast["source_kind"] = "local_official_archive"
breast["label"] = breast["binary_label"]

breast_columns = [
    "dataset", "modality", "task", "image_id", "unit_id", "group_id", "label",
    "released_label", "endpoint_status", "dedup_status", "source_locator",
    "raw_image_sha256", "pixel_sha256", "pixel_width", "pixel_height",
    "pixel_mode", "phash64", "decode_ok",
]
breast_manifest = breast[breast_columns].copy()
write_csv(P2 / "StageT2-J_BrEaST_Harmonised_Exact_Manifest_v0.1.csv", breast_manifest)
write_csv(P3 / "StageT2-J_BrEaST_Internal_Near_Copy_Candidates_v0.1.csv", pd.DataFrame(internal_near_rows))
write_csv(P3 / "StageT2-J_BrEaST_Cross_Roster_Dedup_Candidates_v0.1.csv", pd.DataFrame(cross_rows))

breast_summary = pd.DataFrame([{
    "dataset": "BREAST_LESIONS_USG",
    "released_images": int(len(breast)),
    "endpoint_eligible": int(breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT").sum()),
    "eligible_negative": int(breast["binary_label"].eq(0).sum()),
    "eligible_positive": int(breast["binary_label"].eq(1).sum()),
    "endpoint_excluded": int(~breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT").sum()),
    "exact_or_near_retired_or_held": int(
        breast["dedup_status"].str.contains("EXACT|NEAR_COPY", regex=True).sum()
    ),
    "retained_unique": int(breast["dedup_status"].eq("KEEP_UNIQUE").sum()),
    "retained_groups": int(breast.loc[breast["dedup_status"].eq("KEEP_UNIQUE"), "group_id"].nunique()),
}])
# Correct the endpoint excluded count explicitly.
breast_summary.loc[0, "endpoint_excluded"] = int(
    breast["endpoint_status"].ne("INCLUDE_BINARY_ENDPOINT").sum()
)
write_csv(P3 / "StageT2-J_BrEaST_Dedup_Summary_v0.1.csv", breast_summary)

display(breast_summary)
display(breast["dedup_status"].value_counts().rename_axis("status").reset_index(name="rows"))


,dataset,released_images,endpoint_eligible,eligible_negative,eligible_positive,endpoint_excluded,exact_or_near_retired_or_held,retained_unique,retained_groups
0,BREAST_LESIONS_USG,256,252,154,98,4,0,252,252


,status,rows
0,KEEP_UNIQUE,252
1,EXCLUDE_NORMAL_OR_UNKNOWN,4


In [6]:
# @title T2-J-5. Inventory HiSBreast if present, without outcome mapping
his_root = ACQ_ROOT / "HISBREAST_V2"
his_files = [p for p in his_root.rglob("*") if p.is_file() and p.name != "MANUAL_DROP_HERE.txt"]

his_inventory_rows = []
json_schema_rows = []
diagnosis_strings = []

for path in sorted(his_files):
    relative = str(path.relative_to(his_root))
    his_inventory_rows.append({
        "relative_path": relative,
        "size_bytes": path.stat().st_size,
        "suffix": path.suffix.lower(),
        "sha256": sha_file(path) if path.stat().st_size <= 200 * 1024**2 else "",
        "sha256_status": "COMPUTED" if path.stat().st_size <= 200 * 1024**2 else "DEFERRED_LARGE_FILE",
    })
    if path.suffix.lower() == ".json" and path.stat().st_size <= 20 * 1024**2:
        try:
            value = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            keys = sorted(value.keys()) if isinstance(value, dict) else []
            lower_map = {str(key).lower(): key for key in keys}
            patient_candidates = [
                key for key in keys
                if any(token in str(key).lower() for token in ["patient", "benhnhan", "mabn", "case", "subject"])
            ]
            diagnosis_candidates = [
                key for key in keys
                if any(token in str(key).lower() for token in ["diagn", "chan doan", "chandoan", "ketluan", "result"])
            ]
            for key in diagnosis_candidates:
                diagnosis_strings.append(str(value.get(key, "")))
            json_schema_rows.append({
                "relative_path": relative,
                "keys_json": json.dumps(keys, ensure_ascii=False),
                "patient_candidate_keys_json": json.dumps(patient_candidates, ensure_ascii=False),
                "diagnosis_candidate_keys_json": json.dumps(diagnosis_candidates, ensure_ascii=False),
                "error": "",
            })
        except Exception as exc:
            json_schema_rows.append({
                "relative_path": relative, "keys_json": "[]",
                "patient_candidate_keys_json": "[]",
                "diagnosis_candidate_keys_json": "[]",
                "error": f"{type(exc).__name__}: {exc}",
            })

his_inventory = pd.DataFrame(his_inventory_rows)
his_schema = pd.DataFrame(json_schema_rows)
his_diagnosis = (
    pd.Series([text.strip() for text in diagnosis_strings if text.strip()])
    .value_counts()
    .rename_axis("released_diagnosis_string")
    .reset_index(name="frequency")
)

write_csv(P2 / "StageT2-J_HiSBreast_File_Inventory_v0.1.csv", his_inventory)
write_csv(P2 / "StageT2-J_HiSBreast_JSON_Schema_Audit_v0.1.csv", his_schema)
write_csv(P2 / "StageT2-J_HiSBreast_Released_Diagnosis_String_Inventory_v0.1.csv", his_diagnosis)

his_archive_present = any(
    p.suffix.lower() in {".zip", ".rar", ".7z", ".tar", ".gz"}
    for p in (his_root / "00_Raw_Inbox").glob("*") if p.is_file()
)
his_images_present = sum(
    1 for p in his_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
his_status = (
    "ACQUIRED_SCHEMA_INVENTORY_READY_LABEL_MAP_NOT_AUTHORISED"
    if his_archive_present and his_images_present > 0
    else "HOLD_MANUAL_OFFICIAL_DOWNLOAD_REQUIRED"
)
print("HiSBreast status:", his_status)
print("HiSBreast images present:", his_images_present)
if len(his_diagnosis):
    display(his_diagnosis.head(50))


HiSBreast status: HOLD_MANUAL_OFFICIAL_DOWNLOAD_REQUIRED
HiSBreast images present: 0


In [7]:
# @title T2-J-6. Freeze grouped splits, gates, decision and exact manual instructions
def deterministic_group_split(frame, dataset_id):
    retained = frame[frame["dedup_status"].eq("KEEP_UNIQUE")].copy()
    retained = retained.dropna(subset=["label", "group_id"]).copy()
    group_table = (
        retained.groupby("group_id", as_index=False)
        .agg(label=("label", "first"), images=("image_id", "nunique"))
    )
    conflict = (
        retained.groupby("group_id")["label"].nunique().max()
        if len(retained) else 0
    )
    assert conflict <= 1, f"Group label conflict in {dataset_id}"
    if len(group_table) < 10 or group_table["label"].nunique() < 2:
        return pd.DataFrame(), pd.DataFrame([{
            "dataset": dataset_id, "split_ready": False,
            "reason": "insufficient groups or classes",
        }])

    train_groups, validation_groups = train_test_split(
        group_table,
        test_size=0.20,
        random_state=SEED,
        stratify=group_table["label"],
    )
    group_partition = {
        **{group: "development" for group in train_groups["group_id"]},
        **{group: "validation" for group in validation_groups["group_id"]},
    }
    retained["partition"] = retained["group_id"].map(group_partition)
    no_leakage = retained.groupby("group_id")["partition"].nunique().max() <= 1
    partition_class_counts = retained.groupby("partition")["label"].nunique()
    both_classes = len(partition_class_counts) == 2 and partition_class_counts.min() == 2
    summary = pd.DataFrame([{
        "dataset": dataset_id,
        "retained_images": int(len(retained)),
        "retained_groups": int(retained["group_id"].nunique()),
        "negative_images": int(retained["label"].eq(0).sum()),
        "positive_images": int(retained["label"].eq(1).sum()),
        "development_groups": int(train_groups["group_id"].nunique()),
        "validation_groups": int(validation_groups["group_id"].nunique()),
        "group_leakage_absent": bool(no_leakage),
        "both_classes_each_partition": bool(both_classes),
        "split_ready": bool(no_leakage and both_classes),
        "reason": "",
    }])
    return retained, summary

milk_split, milk_split_summary = deterministic_group_split(milk_manifest, "ISIC_MILK10K")
breast_split, breast_split_summary = deterministic_group_split(breast_manifest, "BREAST_LESIONS_USG")
split_manifest = pd.concat([milk_split, breast_split], ignore_index=True)
split_summary = pd.concat([milk_split_summary, breast_split_summary], ignore_index=True)

write_csv(P4 / "StageT2-J_Frozen_Grouped_Split_Manifest_v0.1.csv", split_manifest)
write_csv(P4 / "StageT2-J_Grouped_Split_Summary_v0.1.csv", split_summary)

milk_counts_exact = (
    int(milk["binary_label"].eq(0).sum()) == 746
    and int(milk["binary_label"].eq(1).sum()) == 450
)
breast_counts_exact = (
    int(breast["binary_label"].eq(0).sum()) == 154
    and int(breast["binary_label"].eq(1).sum()) == 98
    and int(breast["endpoint_status"].ne("INCLUDE_BINARY_ENDPOINT").sum()) == 4
)
fingerprints_complete = (
    milk.loc[milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT"), "decode_ok"].fillna(False).all()
    and breast.loc[breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT"), "decode_ok"].fillna(False).all()
)
split_leakage_absent = bool(
    len(split_summary)
    and split_summary["group_leakage_absent"].fillna(False).all()
)
manual_locations_created = all(
    (ACQ_ROOT / item["dataset_id"] / "00_Raw_Inbox" / "MANUAL_DROP_HERE.txt").is_file()
    for item in MANUAL_QUEUE
)

gates = pd.DataFrame([
    {"gate": "G1_parent_integrity", "passed": True, "observed": "T2-I and T3-PF self-hashes exact"},
    {"gate": "G2_locked_blind_firewall", "passed": True, "observed": "no locked-blind path or asset accessed"},
    {"gate": "G3_bus_uclm_alias_excluded", "passed": True, "observed": "BUS_UCLM_V3 -> BUS_UCLM_2025_V3 existing source"},
    {"gate": "G4_milk_endpoint_mapping", "passed": milk_counts_exact, "observed": f"NV={int(milk['binary_label'].eq(0).sum())}; MEL={int(milk['binary_label'].eq(1).sum())}"},
    {"gate": "G5_breast_endpoint_mapping", "passed": breast_counts_exact, "observed": f"benign={int(breast['binary_label'].eq(0).sum())}; malignant={int(breast['binary_label'].eq(1).sum())}; excluded={int(breast['endpoint_status'].ne('INCLUDE_BINARY_ENDPOINT').sum())}"},
    {"gate": "G6_fingerprints_complete", "passed": fingerprints_complete, "observed": f"MILK={int(milk['decode_ok'].fillna(False).sum())}; BrEaST={int(breast['decode_ok'].fillna(False).sum())}"},
    {"gate": "G7_dedup_policy_applied", "passed": reference_coverage >= MIN_REFERENCE_COVERAGE, "observed": f"dermoscopy reference coverage={reference_coverage:.3f}"},
    {"gate": "G8_grouped_split_leakage_absent", "passed": split_leakage_absent, "observed": split_summary.to_dict("records")},
    {"gate": "G9_manual_drop_locations_created", "passed": manual_locations_created, "observed": f"{len(MANUAL_QUEUE)} exact drop locations"},
    {"gate": "G10_no_performance_scoring", "passed": True, "observed": "no AUC/source-score/RA-CB operation"},
    {"gate": "G11_stage12_false", "passed": t3pf["stage12_authorised"] is False, "observed": t3pf["stage12_authorised"]},
])

ready_targets = split_summary.loc[split_summary["split_ready"].fillna(False), "dataset"].tolist()
integrity_pass = bool(gates.loc[gates["gate"].isin([
    "G1_parent_integrity", "G2_locked_blind_firewall", "G3_bus_uclm_alias_excluded",
    "G4_milk_endpoint_mapping", "G5_breast_endpoint_mapping",
    "G6_fingerprints_complete", "G9_manual_drop_locations_created",
    "G10_no_performance_scoring", "G11_stage12_false",
]), "passed"].all())

if not integrity_pass:
    decision = "TERMINATE_T2J_INTEGRITY_ENDPOINT_OR_FIREWALL_FAILURE"
elif len(ready_targets) == 2:
    decision = "SEAL_TWO_EXPANSION_TARGETS_AUTHORISE_FROZEN_SOURCE_SCORE_PREPARATION_AND_MANUAL_QUEUE"
elif len(ready_targets) == 1:
    decision = "SEAL_ONE_EXPANSION_TARGET_AUTHORISE_PARTIAL_SOURCE_SCORE_PREPARATION_RETAIN_OTHER_HOLD"
else:
    decision = "HOLD_EXPANSION_TARGETS_PENDING_DEDUP_OR_GROUPING_REMEDIATION"

write_csv(P5 / "StageT2-J_Frozen_Gates_v0.1.csv", gates)

completion_core = {
    "stage": "StageT2-J",
    "decision": decision,
    "parent_t2i_final_record_sha256": EXPECTED["t2i_record_sha256"],
    "parent_t3pf_activation_record_sha256": EXPECTED["t3pf_record_sha256"],
    "protocol_seal_sha256": protocol["protocol_seal_sha256"],
    "ready_targets": ready_targets,
    "ready_target_count": len(ready_targets),
    "milk_endpoint_eligible": int(milk["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT").sum()),
    "milk_retained_unique": int(milk["dedup_status"].eq("KEEP_UNIQUE").sum()),
    "milk_reference_fingerprint_coverage": reference_coverage,
    "breast_endpoint_eligible": int(breast["endpoint_status"].eq("INCLUDE_BINARY_ENDPOINT").sum()),
    "breast_retained_unique": int(breast["dedup_status"].eq("KEEP_UNIQUE").sum()),
    "hisbreast_status": his_status,
    "manual_queue_count": len(MANUAL_QUEUE),
    "bus_uclm_alias_excluded": True,
    "gates_passed": int(gates["passed"].sum()),
    "gates_total": int(len(gates)),
    "target_outcomes_scored": False,
    "source_scores_computed": False,
    "method_refit_authorised": False,
    "locked_blind_assets_touched": False,
    "locked_blind_outcomes_accessed": False,
    "stage12_authorised": False,
}
completion = dict(completion_core)
completion["completed_utc"] = now()
completion["final_record_sha256"] = sha_json(completion)
write_json(P5 / "StageT2-J_Complete_v0.1.json", completion)

summary = f"""# Stage T2-J result summary v0.1

- Decision: `{decision}`
- Split-ready expansion targets: `{ready_targets}`
- MILK10K endpoint eligible / retained unique: `{completion['milk_endpoint_eligible']} / {completion['milk_retained_unique']}`
- Existing dermoscopy reference fingerprint coverage: `{reference_coverage:.3f}`
- BrEaST endpoint eligible / retained unique: `{completion['breast_endpoint_eligible']} / {completion['breast_retained_unique']}`
- HiSBreast status: `{his_status}`
- BUS-UCLM alias excluded: `True`
- Manual queue entries: `{len(MANUAL_QUEUE)}`
- Gates: `{int(gates['passed'].sum())}/{len(gates)}`
- Target outcomes scored: `False`
- Locked blind assets touched: `False`
- Stage 12 authorised: `False`
- Final record SHA256: `{completion['final_record_sha256']}`
"""
write_text(P5 / "StageT2-J_Result_Summary_v0.1.md", summary)

display(split_summary)
display(gates)
print("\n========== STAGE T2-J COMPLETE ==========")
print("Decision:", decision)
print("Split-ready targets:", ready_targets)
print("HiSBreast status:", his_status)
print("Manual queue CSV:", P1 / "StageT2-J_Manual_Download_Queue_v0.1.csv")
print("Immediate HiSBreast drop folder:", ACQ_ROOT / "HISBREAST_V2" / "00_Raw_Inbox")
print("Target outcomes scored:", False)
print("Locked blind assets touched:", False)
print("Stage 12 authorised:", False)
print("Final record SHA256:", completion["final_record_sha256"])


,dataset,retained_images,retained_groups,negative_images,positive_images,development_groups,validation_groups,group_leakage_absent,both_classes_each_partition,split_ready,reason
0,ISIC_MILK10K,907,907,564,343,725,182,True,True,True,
1,BREAST_LESIONS_USG,252,252,154,98,201,51,True,True,True,


,gate,passed,observed
0,G1_parent_integrity,True,T2-I and T3-PF self-hashes exact
1,G2_locked_blind_firewall,True,no locked-blind path or asset accessed
2,G3_bus_uclm_alias_excluded,True,BUS_UCLM_V3 -> BUS_UCLM_2025_V3 existing source
3,G4_milk_endpoint_mapping,True,NV=746; MEL=450
4,G5_breast_endpoint_mapping,True,benign=154; malignant=98; excluded=4
5,G6_fingerprints_complete,True,MILK=1196; BrEaST=252
6,G7_dedup_policy_applied,True,dermoscopy reference coverage=1.000
7,G8_grouped_split_leakage_absent,True,"[{'dataset': 'ISIC_MILK10K', 'retained_images'..."
8,G9_manual_drop_locations_created,True,9 exact drop locations
9,G10_no_performance_scoring,True,no AUC/source-score/RA-CB operation



========== STAGE T2-J COMPLETE ==========
Decision: SEAL_TWO_EXPANSION_TARGETS_AUTHORISE_FROZEN_SOURCE_SCORE_PREPARATION_AND_MANUAL_QUEUE
Split-ready targets: ['ISIC_MILK10K', 'BREAST_LESIONS_USG']
HiSBreast status: HOLD_MANUAL_OFFICIAL_DOWNLOAD_REQUIRED
Manual queue CSV: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Cross_Modal/StageT2-J_Expansion_Harmonisation_Dedup_And_Public_Route_Repair_v0.1/01_Route_Repair_And_Manual_Queue/StageT2-J_Manual_Download_Queue_v0.1.csv
Immediate HiSBreast drop folder: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/00_Data_Acquisition/Cross_Modal_Independent_Target_Expansion_v0.1/HISBREAST_V2/00_Raw_Inbox
Target outcomes scored: False
Locked blind assets touched: False
Stage 12 authorised: False
Final record SHA256: dc23137bf52c791bf40132f08bd22396e5d5b6d5d06c886271560161e610f4d8


## Interpretation boundary

- `KEEP_UNIQUE` means the row passed the frozen endpoint and deduplication rules used in this stage.
- A split-ready expansion target may proceed only to a later frozen source-score preparation stage.
- HiSBreast remains unlabelled for this programme until its released diagnosis strings and patient grouping are explicitly mapped and frozen.
- BUS-UCLM cannot increase the target-level sample size because it is already present in the development roster.
- No result from this notebook changes the locked Stage T3 validation protocol.
